In [0]:
from pyspark.sql import functions as F

def process_bronze():
    json_schema = "key string, offset long, partition long, timestamp long, topic string, value string"
    query = (spark.readStream
                 .format("cloudFiles")
                 .option("cloudFiles.format", 'json')
                 .schema(json_schema)
                 .load("/Volumes/workspace/bookstore_eng_pro/dataset/kafka-raw/")
                 .withColumn("timestamp", F.from_unixtime(F.col("timestamp")/1000).cast("timestamp"))
                 .withColumn("year-month", F.date_format(F.col("timestamp"),"yyyy-MM"))
                 .withColumn("_src_file", F.col("_metadata.file_name"))
                 .withColumn("load_timestamp", F.current_timestamp())
                 .writeStream
                 .option("checkpointLocation", "/Volumes/workspace/bookstore_eng_pro/checkpoints/bronze_checkpoint")
                 .option("mergeSchema", "true")
                 .partitionBy("topic", "year-month")
                 .trigger(availableNow=True)
                 .table("workspace.bookstore_eng_pro.bronze")
    )
    query.awaitTermination()



In [0]:
process_bronze()